# Como o Senado vota? Análise das votações nominais de 2025

Este notebook extrai, trata e analisa dados da
[API de Dados Abertos do Senado Federal](https://legis.senado.leg.br/dadosabertos/api-docs/swagger-ui/index.html),
com foco nas **votações nominais do Plenário em 2025**.

## 1. Configuração

Bibliotecas e parâmetros usados em todo o notebook. `PAUSA` espaça as requisições para não
sobrecarregar a API pública.

In [ ]:
import logging
import time
from typing import Any

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ANO: int = 2025
BASE_URL: str = "https://legis.senado.leg.br/dadosabertos"
TIMEOUT: int = 30   # segundos por requisição
PAUSA: float = 0.2  # segundos entre requisições

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("senado")

## 2. Cliente da API

Ao explorar a API, encontramos quatro comportamentos que o cliente precisa tratar:

| Comportamento observado | Tratamento |
|---|---|
| Sem o cabeçalho `Accept: application/json`, a API responde em **XML** | A sessão envia o cabeçalho em toda requisição |
| Um código inexistente (ex.: `/senador/99999999`) retorna **HTTP 200** só com metadados, sem erro | `extrair()` devolve lista vazia quando o caminho não existe |
| Cada endpoint aninha os dados de um jeito (`ListaParlamentarEmExercicio → Parlamentares → Parlamentar`, lista simples em `/votacao`...) e, por herança do XML, **um item único pode vir como objeto em vez de lista** | `extrair()` percorre o caminho informado e sempre devolve `list[dict]` |
| Endpoints documentados que respondem **404** (ex.: `/plenario/votacao/orientacaoBancada/{inicio}/{fim}`) | Erros HTTP viram `ErroAPI`, com mensagem clara |

Além disso, a sessão faz até 5 novas tentativas com *backoff* exponencial em erros transitórios
(`429` e `5xx`).

In [ ]:
class ErroAPI(Exception):
    """Falha ao consultar a API do Senado (rede, HTTP ou resposta inválida)."""


def criar_sessao() -> requests.Session:
    sessao = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,  
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    sessao.mount("https://", HTTPAdapter(max_retries=retry))
    sessao.headers.update({"Accept": "application/json", "User-Agent": "desafio-aleon/1.0"})
    return sessao


def get_json(sessao: requests.Session, endpoint: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{BASE_URL}{endpoint}"
    try:
        resposta = sessao.get(url, params=params, timeout=TIMEOUT)
        resposta.raise_for_status()
        return resposta.json()
    except requests.exceptions.Timeout as e:
        raise ErroAPI(f"Timeout após {TIMEOUT}s: {url}") from e
    except requests.exceptions.RetryError as e:
        raise ErroAPI(f"Tentativas esgotadas: {url}") from e
    except requests.exceptions.ConnectionError as e:
        raise ErroAPI(f"Falha de conexão: {url}") from e
    except requests.exceptions.HTTPError as e:
        raise ErroAPI(f"HTTP {e.response.status_code}: {url}") from e
    except ValueError as e:  
        raise ErroAPI(f"Resposta não é JSON válido: {url}") from e
    finally:
        time.sleep(PAUSA)


def extrair(payload: Any, *caminho: str) -> list[dict]:
    
    no = payload
    for chave in caminho:
        if not isinstance(no, dict) or chave not in no:
            return []
        no = no[chave]
    if no is None:
        return []
    return no if isinstance(no, list) else [no]

### 2.1 Validação do cliente

Quatro testes rápidos: dois casos normais e os dois casos de borda descritos acima.

In [ ]:
sessao = criar_sessao()

# 1. Caso normal com aninhamento profundo
senadores_raw = extrair(get_json(sessao, "/senador/lista/atual"),
                        "ListaParlamentarEmExercicio", "Parlamentares", "Parlamentar")
print(f"Senadores em exercício: {len(senadores_raw)}")

# 2. Caso normal com lista simples
votacoes_raw = get_json(sessao, "/votacao", {"dataInicio": f"{ANO}-01-01", "dataFim": f"{ANO}-12-31"})
print(f"Votações nominais em {ANO}: {len(votacoes_raw)}")

# 3. Código inexistente
inexistente = extrair(get_json(sessao, "/senador/99999999"), "DetalheParlamentar", "Parlamentar")
print(f"Senador inexistente: {inexistente}")

# 4. Endpoint documentado que responde 404
try:
    get_json(sessao, f"/plenario/votacao/orientacaoBancada/{ANO}-03-01/{ANO}-03-31")
except ErroAPI as e:
    print(f"Erro tratado: {e}")

Senadores em exercício: 81
Votações nominais em 2025: 128
Senador inexistente: []
Erro tratado: HTTP 404: https://legis.senado.leg.br/dadosabertos/plenario/votacao/orientacaoBancada/2025-03-01/2025-03-31
